# 01 — Dataset Audit

Phase 1 (see `PHASE_PLAN.txt`). This notebook reads the generated manifest and split files
and documents the dataset before any modelling.

Run from the `notebooks/` directory after:

```powershell
python scripts/audit_dataset.py
python scripts/find_near_duplicates.py
python scripts/build_duplicate_groups.py
python scripts/make_splits.py
```

The raw dataset is never modified by this notebook.

In [ ]:
from pathlib import Path
import json

import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = REPO_ROOT / "data" / "processed"
SPLITS = REPO_ROOT / "data" / "splits"
RAW = REPO_ROOT / "data" / "raw" / "rice_leaf_diseases"

manifest = pd.read_csv(PROCESSED / "dataset_manifest.csv")
audit = json.loads((PROCESSED / "dataset_audit.json").read_text(encoding="utf-8"))
duplicates = json.loads((SPLITS / "duplicate_groups.json").read_text(encoding="utf-8"))
split_manifest = json.loads((SPLITS / "split_manifest.json").read_text(encoding="utf-8"))
test = json.loads((SPLITS / "test.json").read_text(encoding="utf-8"))
cv = json.loads((SPLITS / "cv_folds.json").read_text(encoding="utf-8"))

print("dataset version:", audit["dataset_version"])
print("images:", len(manifest), "| total MB:", round(manifest["bytes"].sum() / 1024 / 1024, 1))

## 1. Class inventory

The approved dataset must contain exactly 40 images per class (120 total).

In [ ]:
counts = manifest.groupby("label").agg(
    images=("path", "count"),
    total_mb=("bytes", lambda values: round(values.sum() / 1024 / 1024, 1)),
)
display(counts)
print("total images:", int(counts["images"].sum()))

## 2. Resolution and aspect ratio

Bacterial Leaf Blight has a single resolution (3081×897) for every image.
Raw resolution must therefore never become a model feature — it would leak the class.
Aspect ratios reach 5.3:1, so segmentation and feature code must handle long thin strips.

In [ ]:
summary = pd.DataFrame(
    {
        label: {
            "images": stats["images"],
            "width": f'{stats["width_min"]}-{stats["width_max"]}',
            "height": f'{stats["height_min"]}-{stats["height_max"]}',
            "aspect": f'{stats["aspect_min"]}-{stats["aspect_max"]}',
            "distinct_resolutions": stats["distinct_resolutions"],
        }
        for label, stats in audit["classes"].items()
    }
).T
display(summary)
print("aspect range overall:", manifest["aspect_ratio"].min(), "-", manifest["aspect_ratio"].max())
print("resolution-specific shortcut check: BLB distinct resolutions =", audit["classes"]["bacterial_leaf_blight"]["distinct_resolutions"])

### Resolution histograms

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(manifest["short_side"], bins=20, color="#1fa380")
axes[0].set_title("Short side (px)")
axes[0].set_xlabel("pixels")
axes[1].hist(manifest["aspect_ratio"], bins=20, color="#515e71")
axes[1].set_title("Aspect ratio (max/min)")
axes[1].set_xlabel("ratio")
plt.tight_layout()
plt.show()


## 3. Tiny images

Nine images have a short side below 128 px (the smallest is 301×71).
A naive 256 px minimum-upload rule would reject 25 of the 120 images, so the Phase 2
quality gate uses a hard reject near 64 px and a soft warning near 224 px.

In [ ]:
tiny = manifest[manifest["short_side"] < 128].sort_values("short_side")
display(tiny[["path", "width", "height", "short_side"]])
print("short side < 128 :", len(tiny))
print("short side < 224 :", int((manifest["short_side"] < 224).sum()))
print("short side < 256 :", int((manifest["short_side"] < 256).sum()))

## 4. Full-frame close-ups (no visible background)

Several images are close-ups where the leaf fills the frame. A leaf detector that expects
a background will fail on them, so Phase 2 needs a documented fallback: if the leaf-like
fraction is above ~0.65, treat the whole frame as the leaf mask.

In [ ]:
for threshold in (0.60, 0.65, 0.70, 0.75, 0.80):
    print(f"leaf-green fraction > {threshold:.2f}: {int((manifest['leaf_green_fraction'] > threshold).sum())}")

close_ups = manifest[manifest["leaf_green_fraction"] > 0.70].sort_values(
    "leaf_green_fraction", ascending=False
)
display(close_ups[["path", "width", "height", "leaf_green_fraction"]])

## 5. EXIF orientation

Ten images store orientation `3` (180°). Preprocessing must apply EXIF orientation before
anything else, otherwise those leaves are analysed upside-down. One of them (`DSC_0367.JPG`)
is in the held-out test set, so this is covered by the evaluation protocol.

In [ ]:
display(manifest["exif_orientation"].value_counts().rename("images").to_frame())
display(manifest[manifest["exif_orientation"] != 1][["path", "width", "height", "exif_orientation"]])

## 6. Quality metric percentiles

These are calibration inputs for `backend/configs/quality_thresholds.yaml` (Phase 2).

> Caveat: global over-exposure is dominated by the white paper background (p95 ≈ 64% of
> pixels). In Phase 2 the over-exposure check must be computed **inside the leaf mask**, not
> over the whole frame.

In [ ]:
quality = pd.DataFrame(audit["quality_metric_percentiles"]).T
display(quality)
print("Blur (Laplacian variance) p5 =", quality.loc["laplacian_variance", "p5"], "-> keep the warn threshold well below it")

## 7. Near-duplicate review

Exact MD5 duplicates: none. The ORB + RANSAC screen followed by visual adjudication found
13 confirmed same-scene groups (27 images, 22.5% of the dataset). All are within a single
class, but they must still never be split across train/validation/test partitions.

Two candidate pairs were rejected as different leaves. Full metrics are in
`data/processed/near_duplicates.csv`; the review sheet is
`data/processed/near_duplicate_review.png`.

In [ ]:
print("groups :", duplicates["summary"]["groups"])
print("images :", duplicates["summary"]["images_in_groups"], duplicates["summary"]["by_class"])

group_rows = [
    {
        "group": group["group_id"],
        "label": group["label"],
        "files": ", ".join(Path(path).name for path in group["paths"]),
    }
    for group in duplicates["confirmed_groups"]
]
display(pd.DataFrame(group_rows))
display(pd.DataFrame(duplicates["rejected_candidates"]))

## 8. Frozen splits

- Held-out test: 24 images, 8 per class, chosen only from singleton images.
- Cross-validation: the other 96 images, stratified 5-fold, group-aware.
- Seed 42; re-running `scripts/make_splits.py` reproduces identical files
  (`python scripts/make_splits.py --verify`).

In [ ]:
print("counts:", split_manifest["counts"])
print("invariants all true:", all(split_manifest["invariants"].values()))

fold_rows = [
    {
        "fold": fold["fold"],
        "train": len(fold["train"]),
        "validation": len(fold["validation"]),
        "validation_counts": fold["validation_counts"],
    }
    for fold in cv["folds"]
]
display(pd.DataFrame(fold_rows))
display(pd.DataFrame(test["items"])[["label", "path"]])

## 9. Visual sample per class

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(16, 8))
for row, label in enumerate(sorted(manifest["label"].unique())):
    sample = manifest[manifest["label"] == label].sample(5, random_state=42)
    for column, (_, item) in enumerate(sample.iterrows()):
        with Image.open(REPO_ROOT / item["path"]) as image:
            axes[row, column].imshow(image.convert("RGB"))
        axes[row, column].set_title(Path(item["path"]).name, fontsize=8)
        axes[row, column].axis("off")
plt.tight_layout()
plt.show()

## 10. Limitations and consequences for later phases

1. No healthy class and no severity ground truth — the API must return `uncertain` rather
   than claim a leaf is healthy, and severity stays an experimental lesion-area estimate.
2. 120 images with 13 duplicate groups — headline accuracy will be optimistic; always report
   cross-validation variation, per-class metrics and the held-out test result.
3. White paper backgrounds dominate exposure statistics — compute quality checks on the
   segmented leaf, not the whole frame.
4. Long thin leaves (up to 5.3:1) and 11 background-free close-ups — segmentation needs
   contour scoring by area/centrality and a full-frame fallback.
5. Ten images carry EXIF orientation 3 — orientation correction is mandatory, not optional.
6. The dataset is photographed under controlled conditions; smartphone photos may behave
   very differently. This belongs in the model card, not in a footnote.